# Notebook 06: A/B Testing and Experimentation

## Why This Matters

A/B testing is how ML models earn the right to go to production. Offline metrics (AUC, NDCG) don't capture all business impacts, and every major tech company—Google, Amazon, Netflix, Airbnb—makes product decisions through randomized controlled experiments. Netflix runs hundreds of A/B tests simultaneously. Amazon has been running experiments since the late 1990s and attributes significant revenue growth to experimentation culture. Understanding the statistical underpinnings of A/B testing (sample size, power, multiple comparisons, novelty effects) and the engineering challenges of running concurrent experiments at scale is expected at senior ML engineering and applied scientist roles.

## 1. The A/B Testing Framework

A/B testing for ML models follows the same statistical framework as product A/B tests, with additional ML-specific considerations: model warm-up, feedback loops, and the gap between offline and online metrics.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(15, 9))
ax.set_xlim(0, 15); ax.set_ylim(0, 9); ax.axis('off')
ax.set_title('ML A/B Test Lifecycle', fontsize=14, fontweight='bold')

def box(ax, x, y, w, h, text, color, fontsize=8.5):
    r = FancyBboxPatch((x,y), w, h, boxstyle="round,pad=0.12",
                       facecolor=color, edgecolor='white', linewidth=1.8, alpha=0.9)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color='white', multialignment='center')

def arr(ax, x1, y1, x2, y2, label=''):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color='#666', lw=1.8))
    if label:
        ax.text((x1+x2)/2, (y1+y2)/2+0.1, label, fontsize=7.5, ha='center', color='#555')

phases = [
    (0.3, 7, 2.7, 1.5, 'Pre-Experiment\nPlanning', '#2C3E50'),
    (3.5, 7, 2.7, 1.5, 'Randomization\n& Assignment', '#2980B9'),
    (6.7, 7, 2.7, 1.5, 'Experiment\nExecution', '#8E44AD'),
    (9.9, 7, 2.7, 1.5, 'Statistical\nAnalysis', '#27AE60'),
    (13.1, 7, 1.5, 1.5, 'Decision\n& Rollout', '#E74C3C'),
]

for x, y, w, h, text, color in phases:
    box(ax, x, y, w, h, text, color)

for i in range(len(phases) - 1):
    x1 = phases[i][0] + phases[i][2]
    x2 = phases[i+1][0]
    y = 7.75
    arr(ax, x1, y, x2, y)

# Phase details
details = [
    (0.3, 5.5, [
        '• Define primary metric',
        '• Set significance α=0.05',
        '• Power analysis: sample size',
        '• Minimum detectable effect',
        '• Guardrail metrics',
    ]),
    (3.5, 5.5, [
        '• Hash user_id → bucket',
        '• Stratify if needed',
        '• Holdback set (if needed)',
        '• Ensure no cross-contamination',
        '• Log assignment events',
    ]),
    (6.7, 5.5, [
        '• Run ≥ 1 full week',
        '• Monitor for SRM',
        '• Watch guardrail metrics',
        '• Don't peek early!',
        '• Log all metric events',
    ]),
    (9.9, 5.5, [
        '• t-test / Mann-Whitney',
        '• Correct for multiple comp.',
        '• Check novelty effects',
        '• Segment analysis',
        '• Practical significance',
    ]),
    (13.1, 5.5, [
        '• Ship / no-ship / iterate',
        '• Gradual rollout',
        '• Document learnings',
        '• Update baseline',
    ]),
]

for x, y, items in details:
    for i, item in enumerate(items):
        ax.text(x + 0.1, y - i * 0.38, item, fontsize=8, color='#444')

ax.text(7.5, 2.5, 'Typical duration: 1-4 weeks | Minimum: 1 full week (avoids day-of-week bias)',
        ha='center', fontsize=9, color='#666', style='italic')

plt.tight_layout()
plt.savefig('ab_test_lifecycle.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Statistical Foundations: Sample Size and Power

Underpowered experiments are the most common mistake in A/B testing. An experiment with insufficient sample size will frequently miss real effects (low power) or produce noisy estimates. The sample size calculation is a key interview question.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

def sample_size_two_proportions(p_control, mde, alpha=0.05, power=0.80):
    """
    Calculate required sample size per arm for a two-proportion test.
    p_control: baseline conversion rate
    mde: minimum detectable effect (absolute, e.g., 0.01 = 1pp lift)
    alpha: significance level (default 0.05)
    power: desired power (default 0.80)
    """
    p_treatment = p_control + mde
    p_pooled = (p_control + p_treatment) / 2
    
    z_alpha = stats.norm.ppf(1 - alpha / 2)  # two-sided
    z_beta = stats.norm.ppf(power)
    
    # Standard formula
    n = ((z_alpha + z_beta) ** 2 * 2 * p_pooled * (1 - p_pooled)) / (mde ** 2)
    return int(np.ceil(n))


# Scenario: testing a new ranking model on CTR
baseline_ctr = 0.05  # 5% baseline CTR

print("Sample Size Requirements for CTR A/B Test")
print(f"Baseline CTR: {baseline_ctr:.1%}, α=0.05, power=0.80")
print("=" * 65)

mdes = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05]
print(f"{'MDE':>8} {'MDE %':>10} {'n per arm':>12} {'Total n':>12} {'Weeks @ 1M DAU':>18}")
print("-" * 65)

for mde in mdes:
    n = sample_size_two_proportions(baseline_ctr, mde)
    relative_lift = mde / baseline_ctr
    weeks_at_1m_dau = (n * 2) / (1_000_000 * 7)  # assuming 50% in experiment, 1M daily users
    print(f"  {mde:.3f} ({mde:.1%}) {relative_lift:>8.0%} lift {n:>12,} {n*2:>12,} {weeks_at_1m_dau:>16.1f}w")

# Visualize power curves
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
mde_range = np.linspace(0.001, 0.05, 100)
for alpha, color, label in [(0.05, '#2980B9', 'α=0.05'), (0.01, '#E74C3C', 'α=0.01')]:
    ns = [sample_size_two_proportions(baseline_ctr, m, alpha=alpha) for m in mde_range]
    ax.plot(mde_range * 100, [n/1e6 for n in ns], color=color, linewidth=2, label=f'{label}, power=0.80')

# Also plot power=0.90
ns_90 = [sample_size_two_proportions(baseline_ctr, m, power=0.90) for m in mde_range]
ax.plot(mde_range * 100, [n/1e6 for n in ns_90], color='#27AE60', linewidth=2, linestyle='--', label='α=0.05, power=0.90')

ax.axvline(x=1, color='gray', linestyle=':', alpha=0.7, label='1pp lift (common target)')
ax.set_xlabel('Minimum Detectable Effect (%)', fontsize=11)
ax.set_ylabel('Sample Size per Arm (millions)', fontsize=11)
ax.set_title('Sample Size vs Detectable Effect', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Statistical significance visualization
ax2 = axes[1]
x = np.linspace(-4, 4, 1000)

# Null distribution
null_dist = stats.norm.pdf(x, 0, 1)
alt_dist = stats.norm.pdf(x, 1.5, 1)  # alternative: true effect

ax2.plot(x, null_dist, 'b-', linewidth=2, label='H₀ (no effect)')
ax2.plot(x, alt_dist, 'g-', linewidth=2, label='H₁ (true effect = δ)')

# Critical value
z_crit = stats.norm.ppf(0.975)
ax2.axvline(x=z_crit, color='red', linestyle='--', linewidth=1.5, label=f'Critical value z={z_crit:.2f}')

# Type I error (false positive)
x_fill_alpha = x[x >= z_crit]
ax2.fill_between(x_fill_alpha, stats.norm.pdf(x_fill_alpha, 0, 1), alpha=0.4, color='red', label='Type I error (α)')

# Power region
ax2.fill_between(x_fill_alpha, stats.norm.pdf(x_fill_alpha, 1.5, 1), alpha=0.3, color='green', label='Power (1-β)')

# Type II error (missed effect)
x_fill_beta = x[x < z_crit]
ax2.fill_between(x_fill_beta, stats.norm.pdf(x_fill_beta, 1.5, 1), alpha=0.3, color='orange', label='Type II error (β)')

ax2.set_xlabel('Test Statistic', fontsize=11)
ax2.set_ylabel('Probability Density', fontsize=11)
ax2.set_title('Type I vs Type II Errors', fontsize=11, fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sample_size_power.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Sample Ratio Mismatch (SRM) Detection

SRM is the silent killer of A/B tests. If the control/treatment split doesn't match the intended ratio, your experiment is compromised—likely by a bug in the assignment mechanism. Every experimentation platform must check for SRM before reporting results.

In [ ]:
import numpy as np
from scipy import stats

def check_srm(n_control: int, n_treatment: int, intended_split: float = 0.5,
              alpha: float = 0.001) -> dict:
    """
    Check for Sample Ratio Mismatch using chi-squared test.
    alpha=0.001 is intentionally strict — SRM is a data quality issue, not a product test.
    """
    n_total = n_control + n_treatment
    expected_control = n_total * (1 - intended_split)
    expected_treatment = n_total * intended_split
    
    observed = [n_control, n_treatment]
    expected = [expected_control, expected_treatment]
    
    chi2, p_value = stats.chisquare(f_obs=observed, f_exp=expected)
    
    actual_split = n_treatment / n_total
    relative_deviation = abs(actual_split - intended_split) / intended_split
    
    srm_detected = p_value < alpha
    
    return {
        'n_control': n_control,
        'n_treatment': n_treatment,
        'intended_split': intended_split,
        'actual_split': actual_split,
        'chi2_statistic': chi2,
        'p_value': p_value,
        'srm_detected': srm_detected,
        'relative_deviation_pct': relative_deviation * 100,
    }


# Test cases
test_cases = [
    ('Healthy 50/50', 500_000, 500_200, 0.5),
    ('Healthy 90/10', 900_500, 100_000, 0.10),
    ('SRM: Bug in assignment', 520_000, 480_000, 0.5),
    ('SRM: Bot traffic in treatment', 500_000, 420_000, 0.5),
    ('SRM: Cookie deletion', 500_000, 465_000, 0.5),
]

print(f"{'Scenario':40s} {'Control':>10} {'Treatment':>10} {'Actual%':>8} {'p-value':>12} {'SRM?':>8}")
print('-' * 95)

for name, n_c, n_t, split in test_cases:
    result = check_srm(n_c, n_t, intended_split=split)
    srm_flag = '⚠ SRM!' if result['srm_detected'] else '✓ OK'
    print(f"{name:40s} {n_c:>10,} {n_t:>10,} {result['actual_split']:>7.1%} "
          f"{result['p_value']:>12.2e} {srm_flag:>8}")

print("\nCommon causes of SRM:")
causes = [
    "Assignment bug: only assigning some user types to treatment",
    "Cookie deletion: treatment users clearing cookies re-enter control",
    "Logging issue: treatment events not being logged",
    "Bot/crawler traffic landing disproportionately in one bucket",
    "Survivorship bias: treatment causes higher churn (users leave study)",
    "Client-side vs server-side assignment mismatch",
]
for c in causes:
    print(f"  ⚠ {c}")

print("\nAction: If SRM detected → DO NOT analyze results. Fix the root cause first.")

## 4. Multiple Comparisons and the Multiple Testing Problem

Testing many metrics simultaneously inflates your false positive rate. If you test 20 metrics at α=0.05, you expect 1 false positive by chance alone. This is why experimentation platforms apply corrections like Bonferroni or Benjamini-Hochberg.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

def bonferroni_correction(p_values, alpha=0.05):
    """Bonferroni: divide alpha by number of tests. Very conservative."""
    corrected_alpha = alpha / len(p_values)
    rejected = [p < corrected_alpha for p in p_values]
    return rejected, corrected_alpha

def benjamini_hochberg(p_values, alpha=0.05):
    """BH correction: controls false discovery rate. Less conservative."""
    n = len(p_values)
    sorted_idx = np.argsort(p_values)
    sorted_p = np.array(p_values)[sorted_idx]
    
    bh_threshold = (np.arange(1, n+1) / n) * alpha
    below = sorted_p <= bh_threshold
    
    if below.any():
        max_rejected_rank = np.where(below)[0][-1]
        rejected_sorted = np.arange(n) <= max_rejected_rank
    else:
        rejected_sorted = np.zeros(n, dtype=bool)
    
    rejected = np.zeros(n, dtype=bool)
    rejected[sorted_idx] = rejected_sorted
    return rejected.tolist()


# Simulate 20 metric tests, 3 truly significant, 17 null
np.random.seed(42)
n_tests = 20
n_true_effects = 3

# Generate p-values: 3 with true effects (small p), 17 null (uniform)
true_p = np.random.uniform(0.001, 0.03, n_true_effects)
null_p = np.random.uniform(0.0, 1.0, n_tests - n_true_effects)
all_p = np.concatenate([true_p, null_p])

metrics = [f'Metric_{i+1:02d}' for i in range(n_tests)]
is_true = [True] * n_true_effects + [False] * (n_tests - n_true_effects)

# Apply corrections
uncorrected = [p < 0.05 for p in all_p]
bonferroni, bon_alpha = bonferroni_correction(all_p)
bh = benjamini_hochberg(all_p)

print(f"{'Metric':12} {'p-value':>10} {'True Effect':>12} {'No Correction':>15} {'Bonferroni':>12} {'BH (FDR)':>10}")
print('-' * 80)

false_positives_uncorrected = 0
for metric, p, true, unc, bon, bh_r in zip(metrics, all_p, is_true, uncorrected, bonferroni, bh):
    if not true and unc:
        false_positives_uncorrected += 1
    flag = '✓' if true else '✗'
    print(f"{metric:12} {p:>10.4f} {flag + ' ' + str(true):>12} "
          f"{'✓ sig' if unc else 'ns':>15} {'✓ sig' if bon else 'ns':>12} {'✓ sig' if bh_r else 'ns':>10}")

print(f"\nSummary:")
print(f"  True effects:           {n_true_effects}")
print(f"  No correction:          {sum(uncorrected)} 'significant' ({false_positives_uncorrected} false positives)")
print(f"  Bonferroni (α={bon_alpha:.4f}): {sum(bonferroni)} 'significant' (very conservative)")
print(f"  Benjamini-Hochberg:     {sum(bh)} 'significant' (best balance for exploratory analysis)")

# False positive inflation
n_metrics_range = np.arange(1, 101)
fp_rate = 1 - (1 - 0.05) ** n_metrics_range  # FWER without correction

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_metrics_range, fp_rate, 'r-', linewidth=2, label='At least one false positive (uncorrected)')
ax.axhline(y=0.05, color='green', linestyle='--', label='α=0.05 target')
ax.axvline(x=20, color='gray', linestyle=':', label='20 metrics')
ax.fill_between(n_metrics_range, fp_rate, 0.05, where=fp_rate > 0.05, alpha=0.15, color='red',
                label='Extra false positives beyond α')
ax.set_xlabel('Number of Metrics Tested', fontsize=11)
ax.set_ylabel('Probability of At Least One False Positive', fontsize=11)
ax.set_title('Multiple Comparisons Problem: Why Correction Is Required', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('multiple_comparisons.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Advanced Experimentation: Beyond Simple A/B Tests

Large ML systems require more sophisticated experimentation techniques: interleaving for rankers, multi-armed bandits for exploration, and holdout sets for long-term effect measurement.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Multi-Armed Bandit: Thompson Sampling simulation
np.random.seed(42)

n_arms = 4
true_ctr = [0.05, 0.06, 0.055, 0.052]  # true CTRs for 4 models
arm_names = ['Model A (control)', 'Model B (best)', 'Model C', 'Model D']

# Thompson Sampling
alpha_params = np.ones(n_arms)  # successes + 1
beta_params = np.ones(n_arms)   # failures + 1
n_per_arm = np.zeros(n_arms)
reward_per_arm = np.zeros(n_arms)

n_rounds = 10_000
arm_selections = np.zeros(n_arms)
cumulative_regret = []
total_reward = 0
optimal_arm = np.argmax(true_ctr)

for t in range(n_rounds):
    # Sample theta from Beta distribution for each arm
    thetas = np.random.beta(alpha_params, beta_params)
    chosen_arm = np.argmax(thetas)
    
    # Observe reward
    reward = np.random.binomial(1, true_ctr[chosen_arm])
    
    # Update
    alpha_params[chosen_arm] += reward
    beta_params[chosen_arm] += (1 - reward)
    n_per_arm[chosen_arm] += 1
    arm_selections[chosen_arm] += 1
    total_reward += reward
    
    # Regret = optimal expected reward - actual reward
    regret = true_ctr[optimal_arm] - true_ctr[chosen_arm]
    cumulative_regret.append(regret)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Arm selection proportions
ax = axes[0]
proportions = arm_selections / n_rounds
colors = ['#E74C3C' if i != optimal_arm else '#27AE60' for i in range(n_arms)]
bars = ax.bar(arm_names, proportions, color=colors, alpha=0.8)
ax.axhline(y=0.25, color='gray', linestyle='--', alpha=0.5, label='Equal allocation (A/B test)')
ax.set_ylabel('Proportion of Traffic', fontsize=10)
ax.set_title('Thompson Sampling:\nTraffic Allocation', fontsize=11, fontweight='bold')
ax.set_xticklabels([n.split('(')[0].strip() for n in arm_names], rotation=15, fontsize=8)
ax.legend(fontsize=8)
ax.grid(True, axis='y', alpha=0.3)
for bar, p in zip(bars, proportions):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{p:.1%}', ha='center', fontsize=9)

# Posterior distributions
ax2 = axes[1]
x = np.linspace(0.03, 0.09, 1000)
colors2 = ['#E74C3C', '#27AE60', '#E67E22', '#8E44AD']
for i, (alpha_p, beta_p, name, color) in enumerate(zip(alpha_params, beta_params, arm_names, colors2)):
    posterior = np.array([np.random.beta(alpha_p, beta_p) for _ in range(10000)])
    ax2.hist(posterior, bins=50, density=True, alpha=0.4, color=color, label=name)
    ax2.axvline(x=true_ctr[i], color=color, linestyle='--', alpha=0.8, linewidth=1.5)

ax2.set_xlabel('CTR', fontsize=10)
ax2.set_ylabel('Density', fontsize=10)
ax2.set_title('Posterior Distributions\n(dashed = true CTR)', fontsize=11, fontweight='bold')
ax2.legend(fontsize=7)
ax2.grid(True, alpha=0.3)

# Cumulative regret
ax3 = axes[2]
cum_regret = np.cumsum(cumulative_regret)
# Compare to random
random_regret = np.cumsum([np.mean([true_ctr[optimal_arm] - ctr for ctr in true_ctr])] * n_rounds)
ax3.plot(cum_regret, color='#2980B9', linewidth=2, label='Thompson Sampling')
ax3.plot(random_regret, color='#E74C3C', linewidth=2, linestyle='--', label='Random (equal allocation)')
ax3.set_xlabel('Round', fontsize=10)
ax3.set_ylabel('Cumulative Regret', fontsize=10)
ax3.set_title('Cumulative Regret Comparison', fontsize=11, fontweight='bold')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

plt.suptitle('Multi-Armed Bandit vs A/B Testing: Explore-Exploit Tradeoff', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('bandit_vs_ab.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Thompson Sampling results after {n_rounds:,} rounds:")
print(f"  Traffic sent to best model (B): {arm_selections[optimal_arm]/n_rounds:.1%}")
print(f"  Total CTR achieved: {total_reward/n_rounds:.4f} (optimal={true_ctr[optimal_arm]:.4f})")
print(f"  Efficiency vs equal split: {(total_reward/n_rounds) / (sum(true_ctr)/4):.3f}x")

## Summary

| Concept | Key Point | Interview Signal |
|---------|-----------|------------------|
| Sample size calculation | n ∝ 1/MDE² × variance; weeks, not days for many products | Can derive or explain the formula |
| Statistical power | 80% standard; 90% for high-stakes; underpowered tests miss real effects | Knows that p-value alone is insufficient |
| SRM detection | Chi-squared test; if SRM present, don't analyze results | Critical quality check often omitted |
| Multiple comparisons | 20 metrics at α=0.05 → ~65% chance of a false positive | Apply Bonferroni or BH correction |
| A/B vs bandits | A/B: fixed allocation, better stats; Bandits: dynamic allocation, less waste | Knows when to use each |
| Novelty effects | Performance may be inflated first 1-2 weeks | Run experiment for ≥1 week before reading results |
| Holdback sets | Keep 1% of users on old model long-term to detect delayed effects | Shows awareness of long-run vs short-run effects |
| Interleaving | Faster ranking evaluation by mixing results from two models | Used at Netflix and LinkedIn for ranker tests |